# Baseline Seq2Seq (No Attention) — Training Notebook

**Model:** 2-layer Bidirectional LSTM Encoder + Unidirectional Decoder (NO attention)  
**Vocab:** SentencePiece BPE 16k–20k  
**Embeddings:** FastText 300d (trained on BPE-tokenised Ubuntu corpus)  
**Purpose:** Assignment Comparison Model 1 (no attention baseline)

**Target BLEU-4 to beat:** 0.1386 (osamadev LSTM+ATTN, greedy, word-level 8k vocab)

---

## Prerequisites
1. Phase 1 pipeline (`new/phase1.py`) must have completed — artifacts in `ARTIFACT_DIR`
2. Google Drive must be mounted at `/content/drive`
3. GPU runtime recommended (T4/A100)

## How to use
Run cells top-to-bottom. Cell 5 auto-skips Phase 1 if artifacts already exist.  
Edit `PROJECT_DIR` in Cell 1 to match your Drive folder name.


In [ ]:
# Cell 1 — Mount Drive + set project paths
from google.colab import drive
drive.mount('/content/drive')

import os

# ── EDIT THIS LINE if your Drive folder is named differently ──
PROJECT_DIR  = '/content/drive/MyDrive/nlp-chatbot-project'

ARTIFACT_DIR = f'{PROJECT_DIR}/new/artifacts'    # phase1 pipeline outputs
CKPT_DIR     = f'{PROJECT_DIR}/new/checkpoints'  # model checkpoints
TB_DIR       = f'{PROJECT_DIR}/new/tb_logs'      # TensorBoard logs

LOG_DIR      = f"{PROJECT_DIR}/new/logs"       # timestamped run logs

for d in (ARTIFACT_DIR, CKPT_DIR, TB_DIR, LOG_DIR):
    os.makedirs(d, exist_ok=True)

print(f"PROJECT_DIR  = {PROJECT_DIR}")
print(f"ARTIFACT_DIR = {ARTIFACT_DIR}")
print(f"CKPT_DIR     = {CKPT_DIR}")
print(f"TB_DIR       = {TB_DIR}")
print("Directories ready ✓")

In [ ]:
# Cell 2 — Install / verify dependencies
# Only runs the pip install when packages are missing (speeds up rerun).
import importlib, subprocess, sys

REQUIRED = [
    "sentencepiece", "gensim", "sacrebleu", "rouge_score",
    "bert_score", "seaborn", "tensorboard",
]

missing = [p for p in REQUIRED if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + missing)
    print(f"Installed: {missing}")
else:
    print("All dependencies present ✓")

In [ ]:
# Cell 3 — Add project to sys.path so `from config import CONFIG` resolves
import sys

for p in (PROJECT_DIR, f'{PROJECT_DIR}/new'):
    if p not in sys.path:
        sys.path.insert(0, p)

# Smoke-check the import
from config import CONFIG, set_seed
print("CONFIG imported ✓  vocab_size =", CONFIG["spm_vocab_size"])

In [ ]:
# Cell 4 — Colab / A100 CONFIG overrides
# These override the relative paths in config.py with absolute Drive paths.
import torch

CONFIG.update({
    # Absolute paths for Colab Drive
    "artifact_dir":          ARTIFACT_DIR,
    "checkpoint_dir":        CKPT_DIR,
    "tensorboard_dir":       TB_DIR,
    "spm_model_path":        f'{ARTIFACT_DIR}/stage5_spm.model',
    "embedding_matrix_path": f'{ARTIFACT_DIR}/stage8_embedding_matrix.npy',

    # Hardware tuning — adjust to your GPU allocation
    "batch_size":       256 if torch.cuda.is_available() else 32,
    "grad_accum_steps": 2,      # effective batch = batch_size * grad_accum_steps
    "num_workers":      4 if torch.cuda.is_available() else 0,
})

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
set_seed(CONFIG["seed"])
print("CONFIG updated ✓")

In [ ]:
# Cell 5 — Run Phase 1 data pipeline (skip if artifacts already exist)
import os
from pathlib import Path

REQUIRED_ARTIFACTS = [
    "stage6_train_ids.jsonl",
    "stage6_val_ids.jsonl",
    "stage6_test_ids.jsonl",
    "stage6_vocab.json",
    "stage8_embedding_matrix.npy",
    "stage5_spm.model",
]

missing_arts = [a for a in REQUIRED_ARTIFACTS
                if not (Path(ARTIFACT_DIR) / a).exists()]

if not missing_arts:
    print("✓ All Phase 1 artifacts present — skipping pipeline")
else:
    print(f"Missing artifacts: {missing_arts}")
    print("Running Phase 1 pipeline (this takes ~30–60 min on Colab) …")
    from phase1 import main as phase1_main
    phase1_main(cfg=CONFIG)
    print("Phase 1 complete ✓")

In [ ]:
# Cell 6 — Launch TensorBoard (optional — run in background)
# %load_ext tensorboard
# %tensorboard --logdir {TB_DIR}
print("Uncomment the two lines above to start TensorBoard")

In [ ]:
# Cell 7 — Train baseline (no-attention) model
from train import train_model

print("=" * 60)
print("TRAINING BASELINE SEQ2SEQ (no attention)")
print("=" * 60)

baseline_history = train_model(
    model_type="baseline",
    config=CONFIG,
    device=device,
    checkpoint_dir=CKPT_DIR,
)

print(f"\nBaseline training complete — {len(baseline_history)} epochs")
best_val = min(e["val_loss"] for e in baseline_history)
print(f"Best val loss: {best_val:.4f}")